<a href="https://colab.research.google.com/github/sreejas7919-bit/DSA_6/blob/main/ev_project_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

An end-to-end ETL pipeline using EV Charging Station and Grid Load data. The pipeline extracts, transforms, cleans, and stores over 5,000 charging records in a SQL database for analysis and reporting.

In [ ]:
import pandas as pd
import os
import shutil

def extract():

    os.makedirs("/content/raw_data", exist_ok=True)

    df = pd.read_csv("/content/EV_Charging_Grid_Load_Dataset_5000.csv")

    print("Number of Records:", df.shape[0])
    print("Number of Columns:", df.shape[1])

    shutil.copy(
        "/content/EV_Charging_Grid_Load_Dataset_5000.csv",
        "/content/raw_data/EV_Charging_Grid_Load_Dataset_5000.csv"
    )

    print("Raw data saved successfully!")

    return df


data = extract()

Number of Records: 5000
Number of Columns: 8
Raw data saved successfully!


In [ ]:
data.head()

,Station_ID,City,Charger_Type,Vehicle_Type,Energy_Consumed_kWh,Grid_Load_kW,Station_Status,Date
0,EV10001,Kochi,Fast,Bus,189.50,333.67,Active,2025-09-19
1,EV10002,Calicut,DC Fast,Bus,112.87,473.04,Active,2025-01-28
2,EV10003,Trivandrum,DC Fast,Car,135.41,242.44,Active,2025-05-27
3,EV10004,Hyderabad,DC Fast,Car,33.96,477.68,Active,2025-05-25
4,EV10005,Kochi,Level 2,Car,185.51,300.70,Active,2025-12-30


In [ ]:
def transform(df):

    df = df.drop_duplicates()

    df = df.dropna()

    df.rename(columns={
        "Station_ID":"StationID",
        "Grid_Load_kW":"GridLoad_kW"
    }, inplace=True)

    df["GridLoad_kW"] = df["GridLoad_kW"].astype(float)

    df["Power_Difference"] = df["GridLoad_kW"] - df["Energy_Consumed_kWh"]

    df["Load_Category"] = df["GridLoad_kW"].apply(
        lambda x: "High" if x > 100 else "Low"
    )

    df = df.sort_values(by="GridLoad_kW", ascending=False)

    df = df[df["Energy_Consumed_kWh"] >= 0]

    return df

In [ ]:
clean_data = transform(data)
clean_data.head()

,StationID,City,Charger_Type,Vehicle_Type,Energy_Consumed_kWh,GridLoad_kW,Station_Status,Date,Power_Difference,Load_Category
4681,EV14682,Hyderabad,Fast,Bus,239.69,499.99,Inactive,2025-06-22,260.30,High
1977,EV11978,Trivandrum,Level 2,Bike,29.72,499.70,Active,2025-02-24,469.98,High
4988,EV14989,Delhi,Slow,Car,214.57,499.58,Inactive,2025-10-09,285.01,High
1508,EV11509,Calicut,Fast,Bus,108.28,499.53,Active,2025-11-21,391.25,High
47,EV10048,Mumbai,DC Fast,Bike,47.99,499.50,Inactive,2025-05-21,451.51,High


In [ ]:
import os

os.makedirs("/content/processed_data", exist_ok=True)

In [ ]:
clean_data.to_csv(
    "/content/processed_data/EV_Charging_Grid_Load_Cleaned.csv",
    index=False
)

In [ ]:
os.listdir("/content/processed_data")

['EV_Charging_Grid_Load_Cleaned.csv']

In [ ]:
import os

os.makedirs("/content/reports", exist_ok=True)

In [ ]:
clean_data.to_csv(
    "/content/reports/Complete_Cleaned_Dataset.csv",
    index=False
)

In [ ]:
top10 = clean_data.head(10)

top10.to_csv(
    "/content/reports/Top10_Records.csv",
    index=False
)

In [ ]:
summary = clean_data.describe()

summary.to_csv(
    "/content/reports/Summary_Statistics.csv"
)

In [ ]:
import os
os.listdir("/content/reports")

['Complete_Cleaned_Dataset.csv', 'Summary_Statistics.csv', 'Top10_Records.csv']